In [48]:
import os
from dotenv import load_dotenv
from google.genai.client import Client 
from google import genai
from google.genai.types import GenerateContentConfig, ThinkingConfig
from pprint import pprint
from pydantic import BaseModel
from enum import Enum

Let's first import our Gemini api key 

In [66]:
load_dotenv()
val = os.getenv("GEMINI_API_KEY")
model = "gemini-2.5-flash"
googleClient = genai.Client()


Now let's define the types that we will need to define our agent . As explained an agent should receive and output messages , but a message could have multiple types. It can be a User message which is the prompt of the user , it can be a System message which is the role that we give to our agent and how he should behave and finally the Agent message which is his answer .

In [ ]:
class MessageType(Enum):
    USER = "user"
    SYSTEM = "system"
    AGENT = "agent"
    
class Message(BaseModel):
    type: MessageType
    content: str

In [71]:
class Agent:
    def __init__(self,client:Client ,systemPrompt:Message):
        
        self.client = client
        self.systemPrompt = systemPrompt
        self.messages = []
        if self.systemPrompt is not None:
            self.messages.append(self.systemPrompt)
    
    # The __call__ function allows as to define what the object should do when its called like this : obj()
    def __call__(self, message : str |None):
        if message:
            self.messages.append(Message(type="user",content=message))
        result = self.execute()
        self.messages.append(Message(type="agent",content=result))
        return result
    
    def execute(self):
        history = [m.to_gemini_format() for m in self.messages]
        
        response = self.client.models.generate_content(
            model="gemini-2.5-flash",
            contents=history
        )
        return response.text

In [72]:
system_instr = Message(type=MessageType.SYSTEM, content="Tu es un assistant sarcastique.")
agent = Agent(client=googleClient, systemPrompt=system_instr)
agent("Salut le boss")


ClientError: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': "Role 'system' is not supported. Please use a valid role: MODEL, USER.", 'status': 'INVALID_ARGUMENT'}}